In [1]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans

In [2]:
random_state = 2025

In [3]:
df_ipipneo_120 = pd.read_csv("../data/kaggle/IPIP120-SCORES.csv")

#### Предобработка

In [4]:
df_ipipneo_120_float_cols = df_ipipneo_120.select_dtypes(include=["float64"]).columns
df_ipipneo_120[df_ipipneo_120_float_cols] = df_ipipneo_120[df_ipipneo_120_float_cols].astype(int)
df_ipipneo_120.columns = df_ipipneo_120.columns.str.strip().str.lower()

print(f"Dataset original total: {len(df_ipipneo_120)}")
print(f"Is there any missing value? {df_ipipneo_120.isnull().values.any()}")
print(f"How many missing values: {df_ipipneo_120.isnull().values.sum()}")

print("Removing invalid data from dataset: IPIP120")
df_ipipneo_120 = df_ipipneo_120[(df_ipipneo_120.loc[:, "i1":"i120"] != 0).all(axis=1)]

df_ipipneo_120 = df_ipipneo_120.dropna(inplace=False)

print(f"New dataset total: {len(df_ipipneo_120)}")
print(f"Number of countries: {len(df_ipipneo_120.country.unique())}")

Dataset original total: 410376
Is there any missing value? True
How many missing values: 208
Removing invalid data from dataset: IPIP120
New dataset total: 410168
Number of countries: 240


## Кластеризация на аспектах личности

In [5]:
X = df_ipipneo_120.copy()

O = X.filter(
    [
        "facet_imagination",
        "facet_artistic_interests",
        "facet_emotionality",
        "facet_adventurousness",
        "facet_intellect",
        "facet_liberalism",
    ]
)
C = X.filter(
    [
        "facet_self_efficacy",
        "facet_orderliness",
        "facet_dutifulness",
        "facet_achievement_striving",
        "facet_self_discipline",
        "facet_cautiousness",
    ]
)
E = X.filter(
    [
        "facet_friendliness",
        "facet_gregariousness",
        "facet_assertiveness",
        "facet_activity_level",
        "facet_excitement_seeking",
        "facet_cheerfulness",
    ]
)
A = X.filter(
    [
        "facet_trust",
        "facet_morality",
        "facet_altruism",
        "facet_cooperation",
        "facet_modesty",
        "facet_sympathy",
    ]
)
N = X.filter(
    [
        "facet_anxiety",
        "facet_anger",
        "facet_depression",
        "facet_self_consciousness",
        "facet_immoderation",
        "facet_vulnerability",
    ]
)

model_facet = pd.concat([O, C, E, A, N], axis=1)
model_facet.head()

,facet_imagination,facet_artistic_interests,facet_emotionality,facet_adventurousness,facet_intellect,facet_liberalism,facet_self_efficacy,facet_orderliness,facet_dutifulness,facet_achievement_striving,...,facet_altruism,facet_cooperation,facet_modesty,facet_sympathy,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability
0,75,40,73,71,59,36,59,32,54,63,...,67,76,74,77,27,11,58,39,96,11
1,64,61,31,88,57,92,89,68,44,16,...,89,4,69,67,58,89,89,57,1,80
2,53,45,3,90,74,90,93,24,14,80,...,2,3,24,12,33,86,29,7,6,15
3,8,60,49,81,86,25,84,65,78,82,...,54,52,38,41,12,57,18,8,44,5
4,53,61,44,41,47,69,49,52,44,16,...,66,82,50,55,21,7,26,38,62,53


In [7]:
kmeans = KMeans(n_clusters=4, random_state=random_state)
kfit = kmeans.fit(model_facet)

predictions = kfit.labels_
model_facet["clusters"] = predictions
model_facet.head()

,facet_imagination,facet_artistic_interests,facet_emotionality,facet_adventurousness,facet_intellect,facet_liberalism,facet_self_efficacy,facet_orderliness,facet_dutifulness,facet_achievement_striving,...,facet_cooperation,facet_modesty,facet_sympathy,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability,clusters
0,75,40,73,71,59,36,59,32,54,63,...,76,74,77,27,11,58,39,96,11,1
1,64,61,31,88,57,92,89,68,44,16,...,4,69,67,58,89,89,57,1,80,2
2,53,45,3,90,74,90,93,24,14,80,...,3,24,12,33,86,29,7,6,15,2
3,8,60,49,81,86,25,84,65,78,82,...,52,38,41,12,57,18,8,44,5,1
4,53,61,44,41,47,69,49,52,44,16,...,82,50,55,21,7,26,38,62,53,2


## Визуализация

In [ ]:
pca = PCA(n_components=2)

df_pca = pd.DataFrame(data=pca.fit_transform(model_facet), columns=["X", "Y"])
df_pca["clusters"] = predictions

df_pca.head()

In [ ]:
plt.figure(figsize=(10, 10))
sns.scatterplot(
    data=df_pca, x="X", y="Y", hue="clusters", legend="full", palette="tab10", alpha=0.8
)
plt.title("Cluster IPIP-NEO Facets | Visualized on PCA")

## Выделение средних

In [35]:
person_trait = ['openness', 'conscientiousness', 'extraversion', 
                                     'agreeableness', 'neuroticism']

In [36]:
person_facet = ['facet_imagination', 'facet_artistic_interests', 'facet_emotionality',
       'facet_adventurousness', 'facet_intellect', 'facet_liberalism',
       'facet_self_efficacy', 'facet_orderliness', 'facet_dutifulness',
       'facet_achievement_striving', 'facet_self_discipline',
       'facet_cautiousness', 'facet_friendliness', 'facet_gregariousness',
       'facet_assertiveness', 'facet_activity_level',
       'facet_excitement_seeking', 'facet_cheerfulness', 'facet_trust',
       'facet_morality', 'facet_altruism', 'facet_cooperation',
       'facet_modesty', 'facet_sympathy', 'facet_anxiety', 'facet_anger',
       'facet_depression', 'facet_self_consciousness', 'facet_immoderation',
       'facet_vulnerability']

In [10]:
model_facet

,facet_imagination,facet_artistic_interests,facet_emotionality,facet_adventurousness,facet_intellect,facet_liberalism,facet_self_efficacy,facet_orderliness,facet_dutifulness,facet_achievement_striving,...,facet_cooperation,facet_modesty,facet_sympathy,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability,clusters
0,75,40,73,71,59,36,59,32,54,63,...,76,74,77,27,11,58,39,96,11,1
1,64,61,31,88,57,92,89,68,44,16,...,4,69,67,58,89,89,57,1,80,2
2,53,45,3,90,74,90,93,24,14,80,...,3,24,12,33,86,29,7,6,15,2
3,8,60,49,81,86,25,84,65,78,82,...,52,38,41,12,57,18,8,44,5,1
4,53,61,44,41,47,69,49,52,44,16,...,82,50,55,21,7,26,38,62,53,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410371,14,61,70,41,37,78,89,60,58,89,...,16,50,79,76,76,68,21,33,53,1
410372,39,87,8,98,92,89,98,40,9,63,...,60,57,66,1,6,11,30,24,1,1
410373,23,61,44,1,37,39,78,81,72,89,...,44,86,79,95,69,89,99,42,97,3
410374,85,40,49,50,59,89,73,32,54,22,...,27,74,66,75,73,92,58,44,76,3


In [12]:
df_ipipneo_120['clusters'] = model_facet['clusters']

C:\Users\dzapl\AppData\Local\Temp\ipykernel_19816\402007784.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ipipneo_120['clusters'] = model_facet['clusters']


In [29]:
df_groupby_cluster = df_ipipneo_120.iloc[:, 130:].groupby('clusters').mean()

#### Выделяем черты

In [39]:
df_groupby_cluster[person_trait].apply(lambda x: round(x, 2))


,openness,conscientiousness,extraversion,agreeableness,neuroticism
clusters,,,,,
0,25.71,70.88,36.23,55.24,45.32
1,47.85,78.84,75.02,68.52,17.36
2,45.63,37.18,69.48,38.64,44.31
3,40.39,26.81,22.13,39.93,78.77


#### Выделяем подчерты

In [38]:
df_groupby_cluster[person_facet]

,facet_imagination,facet_artistic_interests,facet_emotionality,facet_adventurousness,facet_intellect,facet_liberalism,facet_self_efficacy,facet_orderliness,facet_dutifulness,facet_achievement_striving,...,facet_altruism,facet_cooperation,facet_modesty,facet_sympathy,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability
clusters,,,,,,,,,,,,,,,,,,,,,
0,28.660449,34.681378,40.398268,31.887715,36.755174,39.421343,59.655290,65.498031,64.922052,60.555694,...,51.880997,61.463695,52.883153,47.319098,51.619740,40.919495,40.610372,60.794092,35.664191,50.990072
1,36.539456,52.212298,52.289658,59.344094,52.762381,43.889071,74.074486,67.615577,68.834267,71.672860,...,70.151396,66.322640,45.467377,62.266697,26.973788,26.285388,19.462509,26.251488,33.126405,27.092248
2,50.428569,43.347846,46.026271,54.748556,43.362810,51.032820,50.319050,42.284292,35.361753,43.447151,...,50.418355,36.126439,38.431664,47.776525,45.450320,52.984330,37.092177,32.763960,59.821709,48.288315
3,51.282192,42.602707,47.673525,34.455148,41.630920,52.017572,27.928725,39.098678,35.542471,30.872125,...,42.693702,40.895504,59.960386,46.215280,73.623248,62.507208,73.751158,72.194095,61.371790,76.722489


In [43]:
def select_top_facet(df, n_top=5, middle=50):
    top_features = {}
    
    for idx, row in df.iterrows():
        # Вычисляем расстояние от середины для каждого признака
        distance_from_middle = (row - middle).abs()
        
        # Сортируем по удаленности от середины
        sorted_features = distance_from_middle.sort_values(ascending=False)
        
        top_n_distances = sorted_features.head(n_top)
        
        features_info = []
        for feature in top_n_distances.index:
            original_value = row[feature]
            
            features_info.append({
                'feature': feature,
                'value': round(original_value, 2),
            })
        
        top_features[idx] = features_info
    
    return top_features